# Beschreibung: 

# Importe:

In [1]:
import sys
import os

import pandas as pd
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, repo_root)

from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules, compute_coverage

print(f"Pfad zu allen Daten: {repo_root} \nPfad dieser Datei: {os.getcwd()}")

Pfad zu allen Daten: /home/samel/01. Projekte/01. Master/COMPARE_RST 
Pfad dieser Datei: /home/samel/01. Projekte/01. Master/COMPARE_RST/Manuelle_Ausfuehrungen/health


# Daten laden:
Heart Failure Prediction Dataset: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction/data

In [2]:
heart = pd.read_csv(f"{repo_root}/Daten/health/heart_failure.csv")
print(heart.shape)

(918, 12)


# Ausführung:

### Vorbereitung: (Datenaufbereitung)

In [3]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "HeartDisease" # Macht aus dem Kontext heraus am meisten Sinn(Vor allem wenn es sich darum dreht)

In [4]:
cutoffs = {
    "Age": [40, 55, 65],      # jung <40, mittel, erhöht, hoch >65
    "RestingBP": [120, 140],  # normal <120, prähyperton, hyperton >140
    "Cholesterol": [200, 240],  # normal <200, grenzwert, hoch
    "MaxHR": [120, 150],      # niedrige Belastung, normal, hoch
    "Oldpeak": [1, 2],        # leichte ST-Senkung, moderate, stark
}

# Diskretisierung der numerischen Daten:
X = heart.drop(columns=[decision_attr])

# ALLE Konditionsattribute diskretisieren (inkl. kategoriale)
heart_disc = discretize(X, bins=3, cutoffs=cutoffs)

# Entscheidungsattribut wieder anhängen
heart_disc[decision_attr] = heart[decision_attr]
print(heart_disc.columns)

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'Age_disc', 'RestingBP_disc', 'Cholesterol_disc', 'FastingBS_disc',
       'MaxHR_disc', 'Oldpeak_disc', 'HeartDisease'],
      dtype='object')


In [5]:
cond_attrs = []
for col in heart_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte
        elif heart_disc[col].dtype == "object":
            cond_attrs.append(col)
print(cond_attrs)

['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope', 'Age_disc', 'RestingBP_disc', 'Cholesterol_disc', 'FastingBS_disc', 'MaxHR_disc', 'Oldpeak_disc']


### Datenbetrachtung:

In [6]:
reduct, info = quick_reduct(df=heart_disc, attrs=cond_attrs, decision=decision_attr)
rules = induce_rules(heart_disc, reduct, decision_attr)

γ(C) mit allen Attributen: 0.965142
Einzel-γ-Werte:
  Sex: γ = 0.000000
  ChestPainType: γ = 0.000000
  RestingECG: γ = 0.000000
  ExerciseAngina: γ = 0.000000
  ST_Slope: γ = 0.000000
  Age_disc: γ = 0.000000
  RestingBP_disc: γ = 0.000000
  Cholesterol_disc: γ = 0.000000
  FastingBS_disc: γ = 0.000000
  MaxHR_disc: γ = 0.000000
  Oldpeak_disc: γ = 0.000000

Alle γ({a}) = 0, aber γ(C) > 0 → benutze quick_reduct_interaction.


# Resultate

In [7]:
print("Ergebnisse:")

print(f"\n{decision_attr}:", reduct)
print("Anzahl Regeln:", len(rules))

for r in rules[:10]:
    print(r)

#print("Rules:", rules_pass_biased[2]) # Einzeln
#print("Rules:", rules_pass_biased) # Das wären alle
print(dependency(heart_disc, cond_attrs, "HeartDisease"))

Ergebnisse:

HeartDisease: ['Sex', 'ChestPainType', 'Age_disc', 'ST_Slope', 'RestingBP_disc', 'Cholesterol_disc', 'RestingECG', 'MaxHR_disc', 'Oldpeak_disc', 'ExerciseAngina', 'FastingBS_disc']
Anzahl Regeln: 780
{'premise': {'Sex': 'M', 'ChestPainType': 'ATA', 'Age_disc': np.int64(0), 'ST_Slope': 'Up', 'RestingBP_disc': np.int64(1), 'Cholesterol_disc': np.int64(2), 'RestingECG': 'Normal', 'MaxHR_disc': np.int64(2), 'Oldpeak_disc': np.int64(0), 'ExerciseAngina': 'N', 'FastingBS_disc': np.int64(0)}, 'decision': np.int64(0), 'support': 3}
{'premise': {'Sex': 'M', 'ChestPainType': 'ATA', 'Age_disc': np.int64(0), 'ST_Slope': 'Up', 'RestingBP_disc': np.int64(1), 'Cholesterol_disc': np.int64(2), 'RestingECG': 'Normal', 'MaxHR_disc': np.int64(1), 'Oldpeak_disc': np.int64(0), 'ExerciseAngina': 'N', 'FastingBS_disc': np.int64(0)}, 'decision': np.int64(0), 'support': 2}
{'premise': {'Sex': 'M', 'ChestPainType': 'ATA', 'Age_disc': np.int64(0), 'ST_Slope': 'Up', 'RestingBP_disc': np.int64(1), 'Cho